In [ ]:
import os
from collections import defaultdict

import torch
import trimesh
import numpy as np
import networkx as nx
from torch.utils.data import Dataset


class MassHuman(Dataset):
    def __init__(self, root_dir="mass_humans"):
        self.root_dir = root_dir

    def __len__(self):
        return len([file for file in os.listdir(self.root_dir) if file.endswith(".obj")])

    def _load_obj(self, obj_file_path):
        resolver = trimesh.resolvers.FilePathResolver(os.path.dirname(obj_file_path))
        loaded_data = trimesh.load(obj_file_path, force=None, resolver=resolver)
        meshes_to_process = []
        for _, geom in loaded_data.geometry.items():
            if isinstance(geom, trimesh.Trimesh):
                meshes_to_process.append(geom)

        sorted_meshs = sorted(meshes_to_process, key=lambda m: len(m.faces), reverse=True)
        return sorted_meshs[1]

    def _create_face_adjacency_graph(self, mesh):
        vertices = mesh.vertices
        faces = mesh.faces
        coord_to_new_index = {}

        def get_or_add_vertex_index(vertex):
            rounded_coord = tuple(vertex)
            if rounded_coord not in coord_to_new_index:
                coord_to_new_index[rounded_coord] = len(coord_to_new_index)
            return coord_to_new_index[rounded_coord]

        faces_with_merged_vertex_ids = []
        for face in faces:
            merged_vertex_ids = [get_or_add_vertex_index(vertices[j]) for j in face]
            faces_with_merged_vertex_ids.append(merged_vertex_ids)

        merged_vertex_to_original_faces = defaultdict(set)
        for original_face_idx, merged_vertex_ids in enumerate(faces_with_merged_vertex_ids):
            for merged_vertex_id in merged_vertex_ids:
                merged_vertex_to_original_faces[merged_vertex_id].add(original_face_idx)

        G = nx.Graph()
        G.add_nodes_from(range(len(faces)))

        for original_face_set in merged_vertex_to_original_faces.values():
            original_face_list = list(original_face_set)
            for i in range(len(original_face_list)):
                for j in range(i + 1, len(original_face_list)):
                    G.add_edge(original_face_list[i], original_face_list[j])

        return G

    def __getitem__(self, idx):
        eyes_mesh = self._load_obj(f"{self.root_dir}/human_{idx}.obj")
        eyes_mesh.merge_vertices()
        face_adjacency_graph = self._create_face_adjacency_graph(eyes_mesh)
        communities = list(nx.connected_components(face_adjacency_graph))
        sorted_communities = sorted(communities, key=len, reverse=True)
        inner_eyes_faces = sorted_communities[0]
        inner_eyes_faces.update(sorted_communities[1])
        return torch.tensor(list(inner_eyes_faces),dtype=torch.long)


dataset = MassHuman()
dataloader = torch.utils.data.DataLoader(dataset, num_workers=32, shuffle=True, in_order=False)

In [ ]:
from tqdm import tqdm

# Check all samples
his = None
for batch in tqdm(dataloader):
    this = batch[0]
    if his is not None:
        if not torch.equal(this, his):
            raise ValueError("Inconsistent inner-layer-eyes face indices across samples.")
    his = this

np.save("inner_layer_eyes_face_indices.npy", his.numpy())